# Maryland 2024 Data Processing

Produces two output GeoJSON files:
1. `md_blocks.geojson` — census block level
2. `md_precincts.geojson` — precinct level

Both contain: VAP, votes_dem, votes_rep, hisp_vap, white_vap, black_vap, asian_vap

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np

BASE = 'Data/md/'

# ── Load shapefiles ────────────────────────────────────────────────────────────
blocks_gdf = gpd.read_file(BASE + 'blocks/md_2024_gen_2020_blocks.shp')
prec_gdf   = gpd.read_file(BASE + 'precincts/md_2024_gen_all_prec.shp')

# ── Load CSVs ──────────────────────────────────────────────────────────────────
cvap_df = pd.read_csv(BASE + 'cvap/md_cvap_2023_2020_b.csv', dtype={'GEOID20': str})

print('Blocks:', blocks_gdf.shape, '| CRS:', blocks_gdf.crs)
print('Precincts:', prec_gdf.shape, '| CRS:', prec_gdf.crs)
print('CVAP:', cvap_df.shape)

Blocks: (83827, 47) | CRS: EPSG:4269
Precincts: (2020, 50) | CRS: EPSG:32618
CVAP: (83827, 33)


## 1. Block-level file

Join block shapefile (votes + VAP) with CVAP CSV (race) on GEOID20.

In [2]:
# Ensure consistent string type for join key
blocks_gdf['GEOID20'] = blocks_gdf['GEOID20'].astype(str)
cvap_df['GEOID20']    = cvap_df['GEOID20'].astype(str)

# Dem votes = Harris (only major-party Dem presidential candidate)
# Rep votes = Trump
# Sum all congressional + senate Dem/Rep columns too? No — use presidential as the standard

block_cols = [
    'GEOID20', 'VAP_MOD',
    'G24PREDHAR',   # Dem presidential votes
    'G24PRERTRU',   # Rep presidential votes
    'geometry'
]
cvap_cols = ['GEOID20', 'CVAP_TOT23', 'CVAP_HSP23', 'CVAP_WHT23', 'CVAP_BLK23', 'CVAP_ASN23']

blocks_slim = blocks_gdf[block_cols].copy()
cvap_slim   = cvap_df[cvap_cols].copy()

# Left join: keep all blocks; fill missing CVAP with 0
merged_blocks = blocks_slim.merge(cvap_slim, on='GEOID20', how='left')
merged_blocks[cvap_cols[1:]] = merged_blocks[cvap_cols[1:]].fillna(0)

merged_blocks = merged_blocks.rename(columns={
    'VAP_MOD':    'vap',
    'G24PREDHAR': 'votes_dem',
    'G24PRERTRU': 'votes_rep',
    'CVAP_TOT23': 'cvap_total',
    'CVAP_HSP23': 'hisp_vap',
    'CVAP_WHT23': 'white_vap',
    'CVAP_BLK23': 'black_vap',
    'CVAP_ASN23': 'asian_vap',
})

print('Merge coverage:', merged_blocks['hisp_vap'].notna().sum(), '/', len(merged_blocks))
print(merged_blocks[['GEOID20','vap','votes_dem','votes_rep','hisp_vap','white_vap','black_vap','asian_vap']].head())

Merge coverage: 83827 / 83827
           GEOID20  vap  votes_dem  votes_rep  hisp_vap  white_vap  black_vap  \
0  240010001001000   27        3.0       17.0         0         28          0   
1  240010001001001    2        0.0        1.0         0          2          0   
2  240010001001002    4        0.0        2.0         0          3          0   
3  240010001001003    0        0.0        0.0         0          0          0   
4  240010001001004    0        0.0        0.0         0          0          0   

   asian_vap  
0          0  
1          0  
2          0  
3          0  
4          0  


In [3]:
# Sanity checks
print('Total dem votes (blocks):', merged_blocks['votes_dem'].sum())
print('Total rep votes (blocks):', merged_blocks['votes_rep'].sum())
print('Total VAP (blocks):', merged_blocks['vap'].sum())
print('Null geometry rows:', merged_blocks.geometry.isna().sum())

Total dem votes (blocks): 1902572.0
Total rep votes (blocks): 1035547.0
Total VAP (blocks): 4788295
Null geometry rows: 0


In [4]:
# Save block-level GeoJSON
out_blocks = gpd.GeoDataFrame(merged_blocks, geometry='geometry', crs=blocks_gdf.crs)
out_blocks.to_file('Data/md/cleaned/md_blocks.geojson', driver='GeoJSON')
print('Saved md_blocks.geojson')

Saved md_blocks.geojson


## 2. Precinct-level file

- Votes come directly from the precinct shapefile.
- VAP and racial demographics are aggregated from blocks → precincts via spatial join (block centroid → containing precinct).

In [5]:
# Reproject everything to a common CRS before spatial ops
# Use EPSG:4326 (WGS84) — both files should already be in it, but ensure consistency
blocks_proj = blocks_gdf.to_crs('EPSG:4326')
prec_proj   = prec_gdf.to_crs('EPSG:4326')

print('Blocks CRS:', blocks_proj.crs)
print('Precincts CRS:', prec_proj.crs)

Blocks CRS: EPSG:4326
Precincts CRS: EPSG:4326


In [6]:
# Build block centroid GDF with VAP + CVAP fields
block_demo = blocks_proj[['GEOID20', 'VAP_MOD', 'geometry']].copy()
block_demo = block_demo.merge(cvap_slim, on='GEOID20', how='left')
block_demo[cvap_cols[1:]] = block_demo[cvap_cols[1:]].fillna(0)

# Use centroids for point-in-polygon assignment
block_centroids = block_demo.copy()
block_centroids['geometry'] = block_demo.geometry.centroid

print('Block centroids:', len(block_centroids))

Block centroids: 83827


/var/folders/6n/l7qbkb9n3fv51fp2f36sr_x80000gn/T/ipykernel_17113/88636497.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  block_centroids['geometry'] = block_demo.geometry.centroid


In [7]:
# Spatial join: assign each block centroid to a precinct
joined = gpd.sjoin(block_centroids, prec_proj[['UNIQUE_ID', 'geometry']], how='left', predicate='within')

print('Blocks assigned to a precinct:', joined['UNIQUE_ID'].notna().sum())
print('Blocks with no precinct match:', joined['UNIQUE_ID'].isna().sum())

Blocks assigned to a precinct: 82127
Blocks with no precinct match: 1700


In [8]:
# For blocks that fell outside all precincts (gaps/borders), use nearest precinct
unmatched = joined[joined['UNIQUE_ID'].isna()].copy()
print(f'{len(unmatched)} unmatched blocks — assigning to nearest precinct...')

if len(unmatched) > 0:
    nearest = gpd.sjoin_nearest(
        unmatched.drop(columns=['index_right', 'UNIQUE_ID']),
        prec_proj[['UNIQUE_ID', 'geometry']],
        how='left'
    )
    joined.loc[joined['UNIQUE_ID'].isna(), 'UNIQUE_ID'] = nearest['UNIQUE_ID'].values

print('Still unmatched after nearest:', joined['UNIQUE_ID'].isna().sum())

1700 unmatched blocks — assigning to nearest precinct...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Still unmatched after nearest: 0


In [9]:
# Aggregate VAP + CVAP fields to precinct level
agg_cols = ['VAP_MOD', 'CVAP_TOT23', 'CVAP_HSP23', 'CVAP_WHT23', 'CVAP_BLK23', 'CVAP_ASN23']
prec_demo = joined.groupby('UNIQUE_ID')[agg_cols].sum().reset_index()

prec_demo = prec_demo.rename(columns={
    'VAP_MOD':    'vap',
    'CVAP_TOT23': 'cvap_total',
    'CVAP_HSP23': 'hisp_vap',
    'CVAP_WHT23': 'white_vap',
    'CVAP_BLK23': 'black_vap',
    'CVAP_ASN23': 'asian_vap',
})

print('Precinct demo table:', prec_demo.shape)
print(prec_demo.head())

Precinct demo table: (2012, 7)
                   UNIQUE_ID   vap  cvap_total  hisp_vap  white_vap  \
0  Allegany County-:-001-000   678         686        15        675   
1  Allegany County-:-002-000   844         810         0        800   
2  Allegany County-:-003-000   843         763         0        736   
3  Allegany County-:-004-000  6455        6107        84       5742   
4  Allegany County-:-005-000  2012        1995         0       1884   

   black_vap  asian_vap  
0          1          0  
1         10          0  
2          7          0  
3        218         12  
4         92          9  


In [10]:
# Select vote columns from precinct shapefile
prec_vote_cols = [
    'UNIQUE_ID', 'COUNTYFP', 'County', 'Elec_Dist', 'Precinct', 'CONG_DIST', 'SLDL_DIST',
    'G24PREDHAR', 'G24PRERTRU',
    'geometry'
]
prec_slim = prec_proj[prec_vote_cols].copy()
prec_slim = prec_slim.rename(columns={
    'G24PREDHAR': 'votes_dem',
    'G24PRERTRU': 'votes_rep',
})

# Merge vote data with aggregated demographics
merged_prec = prec_slim.merge(prec_demo, on='UNIQUE_ID', how='left')

print('Precinct merge shape:', merged_prec.shape)
print(merged_prec[['UNIQUE_ID','vap','votes_dem','votes_rep','hisp_vap','white_vap','black_vap','asian_vap']].head())

Precinct merge shape: (2020, 16)
                   UNIQUE_ID     vap  votes_dem  votes_rep  hisp_vap  \
0  Allegany County-:-001-000   678.0         72        420      15.0   
1  Allegany County-:-002-000   844.0         82        463       0.0   
2  Allegany County-:-003-000   843.0         83        478       0.0   
3  Allegany County-:-004-000  6455.0       1146       2024      84.0   
4  Allegany County-:-005-000  2012.0        359        685       0.0   

   white_vap  black_vap  asian_vap  
0      675.0        1.0        0.0  
1      800.0       10.0        0.0  
2      736.0        7.0        0.0  
3     5742.0      218.0       12.0  
4     1884.0       92.0        9.0  


In [11]:
# Sanity checks
print('Total dem votes (precincts):', merged_prec['votes_dem'].sum())
print('Total rep votes (precincts):', merged_prec['votes_rep'].sum())
print('Total VAP (precincts):', merged_prec['vap'].sum())

# Compare block totals to precinct totals — should be close
print('\nBlock-level totals for comparison:')
print('  Dem votes:', merged_blocks['votes_dem'].sum())
print('  Rep votes:', merged_blocks['votes_rep'].sum())
print('  VAP:', merged_blocks['vap'].sum())

Total dem votes (precincts): 1902577
Total rep votes (precincts): 1035550
Total VAP (precincts): 4788295.0

Block-level totals for comparison:
  Dem votes: 1902572.0
  Rep votes: 1035547.0
  VAP: 4788295


In [12]:
# Save precinct-level GeoJSON
out_prec = gpd.GeoDataFrame(merged_prec, geometry='geometry', crs='EPSG:4326')
out_prec.to_file('Data/md/cleaned/md_precincts.geojson', driver='GeoJSON')
print('Saved md_precincts.geojson')

Saved md_precincts.geojson


In [13]:
print('Done!')
print('  md_blocks.geojson   —', len(out_blocks), 'census blocks')
print('  md_precincts.geojson —', len(out_prec), 'precincts')
print()
print('Block columns:   ', list(out_blocks.columns))
print('Precinct columns:', list(out_prec.columns))

Done!
  md_blocks.geojson   — 83827 census blocks
  md_precincts.geojson — 2020 precincts

Block columns:    ['GEOID20', 'vap', 'votes_dem', 'votes_rep', 'geometry', 'cvap_total', 'hisp_vap', 'white_vap', 'black_vap', 'asian_vap']
Precinct columns: ['UNIQUE_ID', 'COUNTYFP', 'County', 'Elec_Dist', 'Precinct', 'CONG_DIST', 'SLDL_DIST', 'votes_dem', 'votes_rep', 'geometry', 'vap', 'cvap_total', 'hisp_vap', 'white_vap', 'black_vap', 'asian_vap']
